In [17]:
from pathlib import Path

BOOKS_DIR = Path("/kaggle/input/datasets/mishasavinov/ceramic-net-dataset-books/artifacts_clean/books")
META_CSV_PATH = Path("/kaggle/input/datasets/mishasavinov/ceramic-net-dataset-books/artifacts_clean/meta.csv")
OUTPUT_DIR = Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [18]:
!pip install -q timm pytorch-metric-learning onnx onnxruntime albumentations faiss-cpu

In [19]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from pytorch_metric_learning import losses

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [20]:
df = pd.read_csv(META_CSV_PATH)

def find_image_path(row):
    rel = str(row.get("image_path", "")).strip().replace("\\", "/")
    book = str(row.get("book", "")).strip()
    mark_id = str(row.get("mark_id", "")).strip()
    filename = str(row.get("filename", f"{mark_id}.png")).strip()
    
    candidates = [
        BOOKS_DIR / rel if rel else None,
        BOOKS_DIR / book / "marks" / filename,
        BOOKS_DIR / book / "marks" / f"{mark_id}.png",
        BOOKS_DIR / book / filename,
        BOOKS_DIR / filename
    ]
    for p in candidates:
        if p is not None and p.is_file():
            return str(p)
    return None

df["full_path"] = df.apply(find_image_path, axis=1)
valid_df = df[df["full_path"].notna()].copy().reset_index(drop=True)

valid_df["mark_id_clean"] = valid_df["mark_id"].astype(str).str.strip()
mark_to_id = {mark: idx for idx, mark in enumerate(valid_df["mark_id_clean"].unique())}
valid_df["class_id"] = valid_df["mark_id_clean"].map(mark_to_id)
NUM_CLASSES = len(mark_to_id)

In [21]:
INPUT_SIZE = 224

clean_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

heavy_crop_transform = transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.4, 0.9)),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def enhance_stamp_contrast_np(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    
    if np.mean(gray) < 120:
        gray = 255 - gray
        
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray)
    
    return cv2.cvtColor(enhanced_gray, cv2.COLOR_GRAY2BGR)

def read_and_letterbox_book(path_str):
    img_bgr = cv2.imread(str(path_str))
    if img_bgr is None:
        raise ValueError(f"Не удалось прочитать {path_str}")

    img_bgr = enhance_stamp_contrast_np(img_bgr)

    h, w = img_bgr.shape[:2]
    side = max(h, w)
    square = np.full((side, side, 3), 255, dtype=np.uint8)
    y_off = (side - h) // 2
    x_off = (side - w) // 2
    square[y_off:y_off + h, x_off:x_off + w] = img_bgr

    resized = cv2.resize(square, (INPUT_SIZE, INPUT_SIZE), interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)

In [22]:
from concurrent.futures import ThreadPoolExecutor

paths = valid_df["full_path"].tolist()

with ThreadPoolExecutor(max_workers=4) as executor:
    cached_images = list(executor.map(read_and_letterbox_book, paths))

class FastCachedDataset(Dataset):
    def __init__(self, pil_images, class_ids):
        self.images = pil_images
        self.labels = class_ids

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx: int):
        img_pil = self.images[idx]
        class_id = self.labels[idx]

        view_anchor = clean_transform(img_pil)
        view_fragment = heavy_crop_transform(img_pil)
        return view_anchor, view_fragment, torch.tensor(class_id, dtype=torch.long)

dataset = FastCachedDataset(cached_images, valid_df["class_id"].values)

from torch.utils.data import Subset
from sklearn.model_selection import train_test_split

indices = np.arange(len(dataset))
train_idx, val_idx = train_test_split(indices, test_size=0.15, random_state=42)

train_subset = Subset(dataset, train_idx)
val_subset = Subset(dataset, val_idx)

train_loader = DataLoader(
    train_subset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_subset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

In [23]:
@torch.no_grad()
def evaluate_retrieval(model, loader, device):
    model.eval()
    gallery_embs, query_embs = [], []

    for anchors, fragments, _ in loader:
        anchors = anchors.to(device, non_blocking=True)
        fragments = fragments.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            g_emb = model(anchors)
            q_emb = model(fragments)

        gallery_embs.append(g_emb)
        query_embs.append(q_emb)

    gallery = torch.cat(gallery_embs, dim=0)  # [N_val, D]
    queries = torch.cat(query_embs, dim=0)    # [N_val, D]

    sim_matrix = queries @ gallery.T          # [N_val, N_val]
    targets = torch.arange(len(gallery), device=device)

    top1 = torch.argmax(sim_matrix, dim=1)
    recall_1 = (top1 == targets).float().mean().item()

    k = min(5, len(gallery))
    top5 = torch.topk(sim_matrix, k=k, dim=1).indices
    recall_5 = (top5 == targets.unsqueeze(1)).any(dim=1).float().mean().item()

    return recall_1, recall_5

In [24]:
class DINOv2MetricEncoder(nn.Module):
    def __init__(self, out_dim: int = 384, unfreeze_blocks: int = 1):
        super().__init__()
        self.backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
        for p in self.backbone.parameters():
            p.requires_grad = False
            
        if unfreeze_blocks > 0:
            for p in self.backbone.blocks[-unfreeze_blocks:].parameters():
                p.requires_grad = True
            for p in self.backbone.norm.parameters():
                p.requires_grad = True
                
        self.projector = nn.Sequential(
            nn.Linear(384, 512),
            nn.BatchNorm1d(512),
            nn.PReLU(),
            nn.Linear(512, out_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(x)
        proj = self.projector(feats)
        return F.normalize(proj, p=2, dim=-1)

model = DINOv2MetricEncoder(out_dim=384, unfreeze_blocks=2).to(device)
backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
projector_params = [p for p in model.projector.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 1e-5, "weight_decay": 1e-4},
        {"params": projector_params, "lr": 1e-4, "weight_decay": 1e-4},
    ]
)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 640MB/s]


In [25]:
from pytorch_metric_learning import losses, miners

criterion = losses.SupConLoss(temperature=0.07).to(device)
miner = miners.MultiSimilarityMiner(epsilon=0.1)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

EPOCHS = 30
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda")

best_recall_1 = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for anchors, fragments, labels in train_loader:
        anchors = anchors.to(device, non_blocking=True)
        fragments = fragments.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        combined_images = torch.cat([anchors, fragments], dim=0)
        combined_labels = torch.cat([labels, labels], dim=0)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            embeddings = model(combined_images)
            hard_pairs = miner(embeddings, combined_labels)
            
            if hard_pairs[2].numel() > 0:
                loss = criterion(embeddings, combined_labels, hard_pairs)
            else:
                loss = criterion(embeddings, combined_labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        num_batches += 1

    scheduler.step()
    avg_loss = total_loss / max(num_batches, 1)

    val_r1, val_r5 = evaluate_retrieval(model, val_loader, device)
    
    print(
        f"Epoch {epoch:02d}/{EPOCHS:02d} | "
        f"Train Loss: {avg_loss:.4f} | "
        f"Val Recall@1: {val_r1 * 100:.2f}% | "
        f"Val Recall@5: {val_r5 * 100:.2f}%"
    )

    if val_r1 > best_recall_1:
        best_recall_1 = val_r1
        torch.save(model.state_dict(), OUTPUT_DIR / "stamp_encoder_best.pt")
        print(f"Новая лучшая модель сохранена (Recall@1: {best_recall_1 * 100:.2f}%)")

print(f"\nЛучший Val Recall@1: {best_recall_1 * 100:.2f}%")

Epoch 01/30 | Train Loss: 1.1717 | Val Recall@1: 95.55% | Val Recall@5: 99.55%
Новая лучшая модель сохранена (Recall@1: 95.55%)
Epoch 02/30 | Train Loss: 0.7460 | Val Recall@1: 96.44% | Val Recall@5: 99.78%
Новая лучшая модель сохранена (Recall@1: 96.44%)
Epoch 03/30 | Train Loss: 0.6897 | Val Recall@1: 95.32% | Val Recall@5: 99.33%
Epoch 04/30 | Train Loss: 0.7864 | Val Recall@1: 96.21% | Val Recall@5: 99.55%
Epoch 05/30 | Train Loss: 0.7115 | Val Recall@1: 97.10% | Val Recall@5: 99.55%
Новая лучшая модель сохранена (Recall@1: 97.10%)
Epoch 06/30 | Train Loss: 0.6788 | Val Recall@1: 97.77% | Val Recall@5: 99.78%
Новая лучшая модель сохранена (Recall@1: 97.77%)
Epoch 07/30 | Train Loss: 0.6075 | Val Recall@1: 96.88% | Val Recall@5: 99.55%
Epoch 08/30 | Train Loss: 0.6085 | Val Recall@1: 95.99% | Val Recall@5: 100.00%
Epoch 09/30 | Train Loss: 0.6392 | Val Recall@1: 97.10% | Val Recall@5: 99.78%
Epoch 10/30 | Train Loss: 0.5439 | Val Recall@1: 97.55% | Val Recall@5: 100.00%
Epoch 11/30 

In [26]:
model.eval()
all_embs = []
all_ids = []

with torch.no_grad():
    for idx in range(len(valid_df)):
        row = valid_df.iloc[idx]
        img_pil = read_and_letterbox_book(row["full_path"])
        tensor = clean_transform(img_pil).unsqueeze(0).to(device)
        emb = model(tensor).cpu().numpy().squeeze(0)
        
        all_embs.append(emb)
        all_ids.append(str(row["mark_id_clean"]))

all_embs = np.vstack(all_embs).astype(np.float32)
all_ids = np.array(all_ids)

np.savez_compressed(
    OUTPUT_DIR / "reference_index.npz",
    embeddings=all_embs,
    mark_ids=all_ids
)

In [ ]:
model.eval()
dummy = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE, device=device)
onnx_out = OUTPUT_DIR / "stamp_encoder.onnx"

torch.onnx.export(
    model,
    (dummy,),
    str(onnx_out),
    input_names=["input"],
    output_names=["embedding"],
    dynamo=False,
    dynamic_axes={"input": {0: "batch"}, "embedding": {0: "batch"}},
    opset_version=14,
)